### Bronze | ACGRBR25 (DATASUS) <br>
> **Fonte:** `ACGRBR25.dbc` (DATASUS/SINAN — notificações de acidente de trabalho, CAT, agravo CID-10 Y96, ano 2025) em `/Volumes/laest/bronze/landing_raw/`
> **O que faz:** decodifica o `.dbc` offline (pyreaddbc + dbfread) e grava a tabela `laest.bronze.acgrbr25_raw`
<br>_Autor: Vitor Silva em 31/10/2026_

## O processo — como reproduzir

**Contexto da solução:** o compute serverless bloqueia acesso à internet externa (daí os erros `gaierror` do diagnóstico), e o PySUS 2.x não decodifica `.dbc` local. A saída foi decodificar **offline** o arquivo que já está na landing zone.

1. **Instalação** (`%pip install "pysus<2"` + `%restart_python`). A versão legada traz como dependências `pyreaddbc` (converte `.dbc`→`.dbf`, 100% offline) e `dbfread` (lê DBF em ISO-8859-1) — são elas que fazem todo o trabalho.
2. **Diagnóstico (histórico, opcional)** — células *"Lista as funções"*, *help(pysus)* e *"Reconhecimento do decodificador"*. Mapa do pacote (`pkgutil`), `help()` e busca por funções de decodificação. Documentam como chegamos à solução.
3. **Decodificação e gravação** — célula *"Decodificação e gravação na bronze"*. `dbc2dbf` converte o `.dbc` do volume para `/tmp` — o arquivo original da landing zone **não é alterado** (⚠️ o helper `pysus.data.dbc_to_dbf` apaga o `.dbc` original depois de converter, por isso não o usamos). `dbfread` lê em blocos de 100 mil registros, pandas concatena, e o Spark grava `laest.bronze.acgrbr25_raw` (Delta) com as colunas de rastreio `_source_file` e `_ingested_at`.
4. **Verificação** — célula *"Verificação da bronze"* (contagem, datas extremas e amostra). Resultado esperado: **646.427 registros**.

**Para repetir:** execute as células de código **na ordem, de cima a baixo** (instalação → decodificação → verificação; as células de diagnóstico no meio são opcionais). Após qualquer restart de sessão, reexecute a célula de instalação primeiro.

In [0]:
%pip install "pysus<2"
%restart_python
## Instala o pysus legado (1.x) — que traz as dependências de decodificação
## (pyreaddbc e dbfread) — e reinicia a sessão Python para aplicar

In [0]:
## Lista as funções disponíveis no pacote pysys
import pysus, pkgutil

print("versão:", pysus.__version__)
print([m.name for m in pkgutil.iter_modules(pysus.__path__)])

In [0]:
help(pysus)

In [0]:
# Reconhecimento: localizar o decodificador de .dbc na versão legada (1.x)
# Alguns submódulos fazem chamada de rede no import — por isso o try/except por submódulo
import pkgutil, importlib, inspect

for name in ['pysus.preprocessing', 'pysus.utilities']:
    try:
        pkg = importlib.import_module(name)
    except Exception as e:
        print(f'{name}: FALHOU NO IMPORT ({type(e).__name__})')
        continue
    subs = [m.name for m in pkgutil.iter_modules(pkg.__path__)]
    print(name, '->', subs)
    mods = [pkg]
    for s in subs:
        try:
            mods.append(importlib.import_module(f'{name}.{s}'))
        except Exception as e:
            print(f'  {name}.{s}: FALHOU NO IMPORT ({type(e).__name__})')
    for m in mods:
        for f in dir(m):
            if ('dbc' in f.lower() or 'dbf' in f.lower()) and not f.startswith('_'):
                obj = getattr(m, f)
                if callable(obj):
                    print(f'{m.__name__}.{f}{inspect.signature(obj)}')

In [0]:
# Decodificação ACGRBR25.dbc -> laest.bronze.acgrbr25_raw
# Fluxo: pyreaddbc converte .dbc->dbf (100% local), dbfread lê o DBF (ISO-8859-1),
# Spark grava a bronze. O arquivo original no volume NÃO é alterado (conversão em /tmp).
# Nota: o serverless bloqueia rede externa, por isso o download via pysus.ftp não funciona —
# a decodificação offline do arquivo da landing zone é o caminho.
from pyreaddbc import dbc2dbf
from dbfread import DBF
import pandas as pd, pathlib, datetime
from pyspark.sql import functions as F

SRC = '/Volumes/laest/bronze/landing_raw/ACGRBR25.dbc'
work = pathlib.Path('/tmp/acgrbr25'); work.mkdir(exist_ok=True)
dbf_path = str(work / 'ACGRBR25.dbf')
dbc2dbf(SRC, dbf_path)

dbf = DBF(dbf_path, encoding='iso-8859-1', load=False)
cols = [f.name for f in dbf.fields]
chunks, buf = [], []
for rec in dbf:
    buf.append(rec)
    if len(buf) == 100_000:
        chunks.append(pd.DataFrame(buf, columns=cols).replace('', None)); buf = []
if buf:
    chunks.append(pd.DataFrame(buf, columns=cols).replace('', None))
df = pd.concat(chunks, ignore_index=True) if len(chunks) > 1 else chunks[0]

sdf = (spark.createDataFrame(df)
    .withColumn('_source_file', F.lit('ACGRBR25.dbc'))
    .withColumn('_ingested_at', F.lit(datetime.datetime.now())))
sdf.write.mode('overwrite').format('delta').saveAsTable('laest.bronze.acgrbr25_raw')
print(f'{sdf.count():,} registros gravados em laest.bronze.acgrbr25_raw')

In [0]:
# Verificação da bronze: contagem, datas extremas, municípios e amostra
# Resultado esperado: 646.427 registros (SINAN/CAT, agravo Y96, ano 2025)
display(spark.sql("""
SELECT COUNT(*) AS total,
       COUNT(DISTINCT ID_MUNICIP) AS municipios_notificantes,
       MIN(DT_NOTIFIC) AS primeira_notificacao,
       MAX(DT_NOTIFIC) AS ultima_notificacao
FROM laest.bronze.acgrbr25_raw
"""))
display(spark.sql("SELECT * FROM laest.bronze.acgrbr25_raw LIMIT 5"))